In [ ]:
%matplotlib widget

import numpy as np
import torch as th
import motornet as mn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation

from reaching_model import ReachingModel
from my_utils import run_episode, plot_losses, plot_handpaths, plot_signals

print('All packages imported.')
print('pytorch version: ' + th.__version__)
print('numpy version: ' + np.__version__)
print('motornet version: ' + mn.__version__)

In [ ]:
# LOAD A TRAINED MODEL
# Change the name below to match your saved model directory
MODEL_NAME = "demo_modular_1"

model = ReachingModel.load(MODEL_NAME)
print(model)

In [ ]:
# PLOT TRAINING LOSSES

print(f"training completed {model.training_state.batches_completed} batches")
plot_losses(model.training_state.loss_history)

In [ ]:
# RUN CENTER-OUT TEST

n_tg     = 8    # number of targets for center-out task
sim_time = 3.00 # simulation time (seconds)
FF_k     = 0    # force field strength

n_t = int(sim_time / model.env.dt)
model.task.run_mode = 'test_center_out'

print(f"simulating {model.task.run_mode} with {n_tg} targets")
episode_data = run_episode(model.env, model.task, model.policy, n_tg, n_t, model.device, k=FF_k)

In [ ]:
# Plot hand paths and signals for one trial
plot_handpaths(episode_data=episode_data)
plot_signals(episode_data=episode_data, trial=3)

In [ ]:
# EXTRACT HIDDEN ACTIVITY PER MODULE

h_all = episode_data['hidden'].detach().numpy()
print(f"hidden activity shape: {np.shape(h_all)}")

# Use module_dims from the policy to slice per-module activity
module_names = model.config.module_names
h_modules = {}
for i, name in enumerate(module_names):
    h_modules[name] = h_all[:, :, model.policy.module_dims[i]]
    print(f"{name} hidden activity shape: {np.shape(h_modules[name])}")

In [ ]:
# PCA FOR ALL MODULES (condition-mean subtracted)

n_pcs = 6

pca_results = {}
for name, h_mod in h_modules.items():
    # Subtract condition-mean trajectory at each timestep
    # h_mod shape: (conditions, time, neurons)
    cond_mean = np.mean(h_mod, axis=0, keepdims=True)  # mean across conditions per timestep

    # Range across conditions
    n_range = np.max(h_mod, axis=0, keepdims=True) - np.min(h_mod, axis=0, keepdims=True)

    # Scale
    h_demean = (h_mod - cond_mean) / (n_range + 5)

    # Transpose to (time, condition, neurons) and run PCA
    X = np.transpose(h_demean, (1, 0, 2))
    X_flat = X.reshape(-1, X.shape[-1])
    pca = PCA(n_components=n_pcs)
    X_pca = pca.fit_transform(X_flat)
    X_pca = X_pca.reshape(X.shape[0], X.shape[1], n_pcs)

    pca_results[name] = {'X': X, 'X_pca': X_pca, 'pca': pca}
    print(f"{name}: explained variance (PC1-{n_pcs}) = {pca.explained_variance_ratio_.round(3)}")

In [ ]:
# VARIANCE EXPLAINED PER MODULE (first 6 PCs)

module_names_ordered = model.config.module_names
n_modules = len(module_names_ordered)

fig, axes = plt.subplots(1, n_modules, figsize=(max(4, 3.5 * n_modules), 3), sharey=True, squeeze=False)
axes = axes[0]  # flatten from (1, n_modules) to (n_modules,)

for i, name in enumerate(module_names_ordered):
    var_pct = pca_results[name]['pca'].explained_variance_ratio_[:n_pcs] * 100
    axes[i].bar(np.arange(1, n_pcs + 1), var_pct, color='tab:blue')
    axes[i].set_title(name)
    axes[i].set_xticks(np.arange(1, n_pcs + 1))
    axes[i].set_xlabel('PC')

axes[0].set_ylabel('Variance Explained (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot normalized hidden activity for one module (condition 1)
module_to_plot = module_names_ordered[0]

plt.figure()
plt.plot(pca_results[module_to_plot]['X'][:, 0, :])
plt.title(f'Hidden unit activity — {module_to_plot} (condition 1)')
plt.xlabel('Time (steps)')
plt.ylabel('Activity (normalized)')
plt.show()

In [ ]:
# PC time series for one module
module_to_plot = module_names_ordered[0]
X_pca = pca_results[module_to_plot]['X_pca']

time = np.arange(X_pca.shape[0])
colors = plt.cm.tab10(np.linspace(0, 1, X_pca.shape[1]))

fig, axes = plt.subplots(3, 1, figsize=(6, 8), sharex=True)

for i, ax in enumerate(axes):
    for cond in range(X_pca.shape[1]):
        ax.plot(time, X_pca[:, cond, i], color=colors[cond], label=f"Dir {cond+1}" if i==0 else None)
    ax.set_ylabel(f"PC{i+1}")
    ax.set_title(f"PC {i+1} — {module_to_plot}")

axes[-1].set_xlabel("Time")
axes[0].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ANIMATED PCA

go_times = episode_data['delay_go_times']
tg_times = episode_data['delay_tg_times']
dt = episode_data['dt']

def animate_module(name):
    X_pca = pca_results[name]['X_pca']
    n_conds = X_pca.shape[1]
    colors = plt.cm.tab10(np.linspace(0, 1, n_conds))

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_title(name)

    mins = X_pca.min(axis=(0, 1))
    maxs = X_pca.max(axis=(0, 1))
    max_range = (maxs - mins).max()
    padding = max_range * 0.05
    mids = (mins + maxs) / 2
    for j in range(3):
        lo, hi = mids[j] - max_range / 2 - padding, mids[j] + max_range / 2 + padding
        [ax.set_xlim, ax.set_ylim, ax.set_zlim][j](lo, hi)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')

    lines = [ax.plot([], [], [], color=colors[c], label=f"Dir {c+1}")[0] for c in range(n_conds)]
    markers = [ax.scatter([], [], [], color=colors[c], s=60, zorder=5) for c in range(n_conds)]
    go_marks = [ax.scatter([], [], [], color=colors[c], marker='s', s=40,
                           edgecolors='black', linewidths=1, zorder=6) for c in range(n_conds)]
    tg_marks = [ax.scatter([], [], [], color=colors[c], marker='o', s=30,
                           edgecolors='black', linewidths=1, zorder=6) for c in range(n_conds)]
    time_text = ax.text2D(0.02, 0.95, '', transform=ax.transAxes, fontsize=12)
    ax.legend(loc='upper right')

    def animate(frame):
        for c in range(n_conds):
            lines[c].set_data(X_pca[:frame+1, c, 0], X_pca[:frame+1, c, 1])
            lines[c].set_3d_properties(X_pca[:frame+1, c, 2])
            markers[c]._offsets3d = ([X_pca[frame, c, 0]],
                                     [X_pca[frame, c, 1]],
                                     [X_pca[frame, c, 2]])
            if frame >= tg_times[c]:
                tg_marks[c]._offsets3d = ([X_pca[tg_times[c], c, 0]],
                                          [X_pca[tg_times[c], c, 1]],
                                          [X_pca[tg_times[c], c, 2]])
            if frame >= go_times[c]:
                go_marks[c]._offsets3d = ([X_pca[go_times[c], c, 0]],
                                          [X_pca[go_times[c], c, 1]],
                                          [X_pca[go_times[c], c, 2]])
        time_text.set_text(f't = {frame * dt:.2f}s')
        return lines + markers + go_marks + tg_marks + [time_text]

    anim = FuncAnimation(fig, animate, frames=X_pca.shape[0],
                         interval=30, blit=False)
    plt.show()
    return anim


In [ ]:
# ANIMATED PCA — ALL MODULES
anims = {}
for name in module_names_ordered:
    anims[name] = animate_module(name)

In [ ]:
# SAVE ANIMATIONS TO MP4

from matplotlib.animation import FFMpegWriter

writer = FFMpegWriter(fps=30, metadata=dict(artist='motornet_demo'))

for name, anim in anims.items():
    fname = f"visualize_modular_pca_{name}.mp4"
    print(f"Saving {fname} ...")
    anim.save(fname, writer=writer)
    print(f"  done.")

print("All animations saved.")

In [ ]:
# COMBINED ANIMATION — ALL MODULES

from matplotlib.animation import FFMpegWriter

n_mods = len(module_names_ordered)
first_name = module_names_ordered[0]
n_conds = pca_results[first_name]['X_pca'].shape[1]
colors = plt.cm.tab10(np.linspace(0, 1, n_conds))
n_frames = pca_results[first_name]['X_pca'].shape[0]

fig = plt.figure(figsize=(max(6, 5.5 * n_mods), 5))
axes, all_lines, all_markers, all_go_marks, all_tg_marks, all_X_pca = [], [], [], [], [], []

for idx, name in enumerate(module_names_ordered):
    ax = fig.add_subplot(1, n_mods, idx + 1, projection='3d')
    X_pca = pca_results[name]['X_pca']
    all_X_pca.append(X_pca)

    ax.set_title(name, fontsize=14, fontweight='bold')

    # Set equal axis limits
    mins = X_pca.min(axis=(0, 1))
    maxs = X_pca.max(axis=(0, 1))
    max_range = (maxs - mins).max()
    padding = max_range * 0.05
    mids = (mins + maxs) / 2
    for j in range(3):
        lo, hi = mids[j] - max_range / 2 - padding, mids[j] + max_range / 2 + padding
        [ax.set_xlim, ax.set_ylim, ax.set_zlim][j](lo, hi)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')

    lines = [ax.plot([], [], [], color=colors[c], label=f"Dir {c+1}")[0] for c in range(n_conds)]
    markers = [ax.scatter([], [], [], color=colors[c], s=40, zorder=5) for c in range(n_conds)]
    go_marks = [ax.scatter([], [], [], color=colors[c], marker='s', s=60,
                           edgecolors='black', linewidths=1, zorder=6) for c in range(n_conds)]
    tg_marks = [ax.scatter([], [], [], color=colors[c], marker='o', s=30,
                           edgecolors='black', linewidths=1, zorder=6) for c in range(n_conds)]

    if idx == 0:
        ax.legend(loc='upper right', fontsize=8)

    axes.append(ax)
    all_lines.append(lines)
    all_markers.append(markers)
    all_go_marks.append(go_marks)
    all_tg_marks.append(tg_marks)

time_text = fig.text(0.5, 0.02, '', ha='center', fontsize=14)
fig.suptitle('Neural State Trajectories (PCA)', fontsize=16, fontweight='bold')
fig.subplots_adjust(left=0.05, right=0.95, top=0.88, bottom=0.08, wspace=0.15)

def animate_all(frame):
    for idx in range(n_mods):
        X_pca = all_X_pca[idx]
        for c in range(n_conds):
            all_lines[idx][c].set_data(X_pca[:frame+1, c, 0], X_pca[:frame+1, c, 1])
            all_lines[idx][c].set_3d_properties(X_pca[:frame+1, c, 2])
            all_markers[idx][c]._offsets3d = ([X_pca[frame, c, 0]],
                                              [X_pca[frame, c, 1]],
                                              [X_pca[frame, c, 2]])
            if frame >= tg_times[c]:
                all_tg_marks[idx][c]._offsets3d = ([X_pca[tg_times[c], c, 0]],
                                                   [X_pca[tg_times[c], c, 1]],
                                                   [X_pca[tg_times[c], c, 2]])
            if frame >= go_times[c]:
                all_go_marks[idx][c]._offsets3d = ([X_pca[go_times[c], c, 0]],
                                                   [X_pca[go_times[c], c, 1]],
                                                   [X_pca[go_times[c], c, 2]])
    time_text.set_text(f't = {frame * dt:.2f}s')

anim_combined = FuncAnimation(fig, animate_all, frames=n_frames, interval=30, blit=False)

writer = FFMpegWriter(fps=30, metadata=dict(artist='motornet_demo'))
fname = 'visualize_modular_pca_combined.mp4'
print(f"Saving {fname} ...")
anim_combined.save(fname, writer=writer)
print("Done.")

plt.show()